# DurakZero Interactive Play

Use this notebook to explore DurakZero outside of the training loop. It provides tools for

* inspecting or creating custom starting deals,
* watching full self-play games rendered as text, and
* playing complete matches against the current DurakZero policy (or a random baseline).


> **Tip:** The code automatically falls back to a random policy when no model checkpoint is supplied. Set `CHECKPOINT_PATH` below to load a trained DurakZero agent.


In [ ]:
import numpy as np
import torch
from types import SimpleNamespace

from douzero.env import Env
from douzero.env.env import (
    ACTION_END_ATTACK,
    ACTION_TAKE_CARDS,
    NUM_CARDS,
    RANKS,
    SUITS,
    DurakState,
    card_rank,
    card_suit,
    card_to_id,
    id_to_card,
    initial_state,
    _choose_initial_attacker,
    _compute_attack_limit,
)
from douzero.dmc.models import Model


In [ ]:
SUIT_SYMBOLS = ['♣', '♦', '♥', '♠']
ALT_SUIT_SYMBOLS = {'C': '♣', 'D': '♦', 'H': '♥', 'S': '♠'}
SUIT_SYMBOL_TO_INDEX = {sym: idx for idx, sym in enumerate(SUIT_SYMBOLS)}

RANK_LABELS = []
RANK_LABEL_TO_INDEX = {}
for idx, rank_name in enumerate(RANKS):
    label = 'T' if rank_name == '10' else rank_name.upper()
    RANK_LABELS.append(label)
    RANK_LABEL_TO_INDEX[label] = idx
    RANK_LABEL_TO_INDEX[rank_name.upper()] = idx
    if rank_name == '10':
        RANK_LABEL_TO_INDEX['10'] = idx


def card_id_to_string(card_id: int) -> str:
    suit_idx, rank_idx = id_to_card(card_id)
    return f"{RANK_LABELS[rank_idx]}{SUIT_SYMBOLS[suit_idx]}"


def parse_card_string(card_str: str) -> int:
    s = card_str.strip()
    if len(s) < 2:
        raise ValueError(f"Cannot parse card from '{card_str}'.")
    suit_symbol = s[-1]
    rank_token = s[:-1].upper()
    if suit_symbol.upper() in ALT_SUIT_SYMBOLS:
        suit_symbol = ALT_SUIT_SYMBOLS[suit_symbol.upper()]
    if suit_symbol not in SUIT_SYMBOL_TO_INDEX:
        raise ValueError(f"Unknown suit symbol '{suit_symbol}' in '{card_str}'.")
    if rank_token == '10':
        rank_token = 'T'
    if rank_token not in RANK_LABEL_TO_INDEX:
        raise ValueError(f"Unknown rank token '{rank_token}' in '{card_str}'.")
    suit_idx = SUIT_SYMBOL_TO_INDEX[suit_symbol]
    rank_idx = RANK_LABEL_TO_INDEX[rank_token]
    return card_to_id((suit_idx, rank_idx))


def hand_to_string(hand):
    if not hand:
        return '(empty)'
    return ' '.join(card_id_to_string(card) for card in sorted(hand))


def table_to_string(table):
    if not table:
        return '(empty)'
    entries = []
    for attack, defense in table:
        attack_str = card_id_to_string(attack)
        if defense is None:
            entries.append(attack_str)
        else:
            entries.append(f"{attack_str}→{card_id_to_string(defense)}")
    return ' | '.join(entries)


def action_to_string(action_id: int) -> str:
    if 0 <= action_id < NUM_CARDS:
        return card_id_to_string(action_id)
    if action_id == ACTION_END_ATTACK:
        return 'End attack'
    if action_id == ACTION_TAKE_CARDS:
        return 'Take cards'
    raise ValueError(f'Invalid action id: {action_id}')


In [ ]:
DEVICE = 0 if torch.cuda.is_available() else 'cpu'


def _device_string(device):
    return 'cpu' if device == 'cpu' else f'cuda:{device}'


def load_trained_model(checkpoint_path: str, device=DEVICE) -> Model:
    model = Model(device=device)
    map_location = _device_string(device)
    state = torch.load(checkpoint_path, map_location=map_location)
    model_state = state.get('model_state_dict', state)
    for position in ['player_0', 'player_1']:
        if position in model_state:
            model.get_model(position).load_state_dict(model_state[position])
    model.eval()
    return model


def choose_action(obs, model=None, device=DEVICE, epsilon: float = 0.0, rng=None):
    legal_actions = obs['legal_actions']
    if rng is None:
        rng = np.random.default_rng()
    if model is None:
        action_index = int(rng.integers(len(legal_actions)))
        action_id = int(legal_actions[action_index])
        return action_id, action_index, None
    device_str = _device_string(device)
    state_tensor = torch.from_numpy(obs['state']).to(device_str)
    action_embeddings = torch.from_numpy(obs['action_embeddings']).to(device_str)
    flags = SimpleNamespace(exp_epsilon=epsilon)
    with torch.no_grad():
        agent_output = model.act(obs['position'], state_tensor, action_embeddings, flags=flags)
    action_index = agent_output['action_index']
    action_id = int(legal_actions[action_index])
    return action_id, action_index, agent_output


In [ ]:
def run_self_play_episode(model=None, seed: int | None = None, max_steps: int = 512):
    env = Env(seed=seed)
    obs = env.reset()
    rng = np.random.default_rng(seed)
    log = []
    step = 0
    done = False
    while not done and step < max_steps:
        state = env.state
        assert state is not None
        current_player = state.attacker if state.phase == 'attack' else state.defender
        action_id, _, _ = choose_action(obs, model=model, device=DEVICE, rng=rng)
        log.append({
            'step': step,
            'player': current_player,
            'phase': state.phase,
            'attacker': state.attacker,
            'defender': state.defender,
            'hand_strings': [hand_to_string(state.hands[0]), hand_to_string(state.hands[1])],
            'table': table_to_string(state.table),
            'action': action_to_string(action_id),
            'talon_count': len(state.talon),
            'trump_card': card_id_to_string(state.trump_card),
            'trump_suit': SUITS[state.trump_suit],
        })
        obs, reward, done, info = env.step(action_id)
        if done:
            log.append({'result': 'completed', 'winner': info.get('winner')})
        elif obs is None:
            break
        step += 1
    env.close()
    return log


def print_self_play_log(log):
    for entry in log:
        if 'result' in entry:
            winner = entry.get('winner')
            if winner is None:
                print('Game finished without a declared winner.')
            else:
                print(f"Game finished: Player {winner} wins.")
            continue
        role = 'attacking' if entry['phase'] == 'attack' else 'defending'
        print(f"Turn {entry['step']:02d} | Player {entry['player']} ({role}) plays {entry['action']}")
        print(f"    Attacker: P{entry['attacker']}  Defender: P{entry['defender']}")
        print(f"    Table: {entry['table']}")
        print(f"    Trump card: {entry['trump_card']} (suit {entry['trump_suit']})")
        print(f"    Hands: P0 [{entry['hand_strings'][0]}] | P1 [{entry['hand_strings'][1]}]")
        print(f"    Talon remaining: {entry['talon_count']}")
        print()


In [ ]:
CHECKPOINT_PATH = None  # e.g. 'checkpoints/durakzero_latest.pth'
MODEL = load_trained_model(CHECKPOINT_PATH, device=DEVICE) if CHECKPOINT_PATH else None

self_play_log = run_self_play_episode(model=MODEL, seed=2024)
print_self_play_log(self_play_log)


In [ ]:
def create_custom_initial_state(player_cards, human_player: int = 0, seed: int | None = None) -> DurakState:
    if human_player not in (0, 1):
        raise ValueError('DurakZero currently supports exactly two players (0 or 1).')
    rng = np.random.default_rng(seed)
    parsed_cards = [parse_card_string(card) for card in player_cards]
    if len(set(parsed_cards)) != len(parsed_cards):
        raise ValueError('Duplicate cards were provided.')
    if len(parsed_cards) > 6:
        raise ValueError('A hand may not contain more than six cards at the start of the game.')
    deck = list(range(NUM_CARDS))
    for card in parsed_cards:
        deck.remove(card)
    rng.shuffle(deck)
    hands = [[] for _ in range(2)]
    hands[human_player] = parsed_cards.copy()
    while len(hands[human_player]) < 6:
        hands[human_player].append(deck.pop())
    hands[human_player].sort()
    opponent = 1 - human_player
    while len(hands[opponent]) < 6:
        hands[opponent].append(deck.pop())
    hands[opponent].sort()
    trump_card = deck.pop()
    trump_suit = card_suit(trump_card)
    talon = deck
    talon.append(trump_card)
    attacker = _choose_initial_attacker(hands, trump_suit)
    defender = 1 - attacker
    state = DurakState(
        hands=hands,
        talon=talon,
        discard=[],
        table=[],
        attacker=attacker,
        defender=defender,
        phase='attack',
        trump_suit=trump_suit,
        trump_card=trump_card,
        seen_cards={trump_card},
        round_attack_limit=_compute_attack_limit(hands[defender]),
        last_round_winner=None,
    )
    return state


def preview_custom_state(state: DurakState, human_player: int = 0):
    trump_str = card_id_to_string(state.trump_card)
    attacker_role = 'attacker' if state.attacker == human_player else 'defender'
    print(f"Trump card: {trump_str} (suit {SUITS[state.trump_suit]})")
    print(f"You are Player {human_player} and start as the {attacker_role}.")
    print(f"Your hand: {hand_to_string(state.hands[human_player])}")
    print(f"Opponent hand size: {len(state.hands[1 - human_player])} (hidden)")
    print(f"Talon cards remaining: {len(state.talon)}")
    print(f"Initial attack limit against the defender: {state.round_attack_limit}")


In [ ]:
custom_state = create_custom_initial_state(
    ['6♣', '7♦', '8♥', '9♠', 'T♣', 'A♦'],
    human_player=0,
    seed=123,
)
preview_custom_state(custom_state, human_player=0)


In [ ]:
def _prompt_action(legal_actions):
    while True:
        raw = input("Select action (index, card like J♠, 'end', or 'take'): ").strip()
        if raw.isdigit():
            idx = int(raw)
            if 0 <= idx < len(legal_actions):
                return int(legal_actions[idx])
        lowered = raw.lower()
        if lowered in {'end', 'pass', 'done'}:
            candidate = ACTION_END_ATTACK
            if candidate in legal_actions:
                return candidate
        if lowered in {'take', 'pickup', 'pick'}:
            candidate = ACTION_TAKE_CARDS
            if candidate in legal_actions:
                return candidate
        try:
            candidate = parse_card_string(raw)
            if candidate in legal_actions:
                return candidate
        except ValueError:
            pass
        print('Invalid choice, please try again.')


def play_interactive_match(
    model=None,
    device=DEVICE,
    seed: int | None = None,
    custom_state: DurakState | None = None,
    human_player: int = 0,
    epsilon: float = 0.0,
):
    env = Env(seed=seed)
    if custom_state is None:
        obs = env.reset()
    else:
        env.state = custom_state.copy()
        obs = env._build_observation()
    rng = np.random.default_rng(seed)
    done = False
    while not done:
        state = env.state
        assert state is not None
        current_player = state.attacker if state.phase == 'attack' else state.defender
        print('
' + '-' * 60)
        print(f"Current phase: {state.phase} | Attacker: P{state.attacker} | Defender: P{state.defender}")
        print(f"Trump card: {card_id_to_string(state.trump_card)} (suit {SUITS[state.trump_suit]})")
        print(f"Table: {table_to_string(state.table)}")
        print(f"Talon remaining: {len(state.talon)}")
        print(f"Hand sizes -> P0: {len(state.hands[0])}, P1: {len(state.hands[1])}")
        if current_player == human_player:
            print(f"Your hand: {hand_to_string(state.hands[human_player])}")
            for idx, action_id in enumerate(obs['legal_actions']):
                print(f"  [{idx}] {action_to_string(int(action_id))}")
            action_id = _prompt_action(obs['legal_actions'])
            print(f"You play: {action_to_string(action_id)}")
        else:
            action_id, _, _ = choose_action(obs, model=model, device=device, epsilon=epsilon, rng=rng)
            print(f"Model (Player {current_player}) plays: {action_to_string(action_id)}")
        obs, reward, done, info = env.step(action_id)
        if done:
            winner = info.get('winner')
            if winner == human_player:
                print(f"
Game over: you win as Player {winner}!")
            else:
                print(f"
Game over: Player {winner} wins.")
        elif obs is None:
            obs = env._build_observation()
    env.close()


In [ ]:
# Example: play from a fresh random deal.
# To challenge a trained policy, set CHECKPOINT_PATH above.
play_interactive_match(model=MODEL, device=DEVICE, seed=2025)


In [ ]:
# Example: start from the custom position defined earlier.
# Uncomment the following line to try it.
# play_interactive_match(model=MODEL, device=DEVICE, custom_state=custom_state, seed=2025)
